# Rico to YOLO Preprocessing

## Objective

The Rico dataset stores user interface (UI) screens as images with hierarchical JSON annotations.

However, YOLO models cannot directly use these JSON files.

The objective of this notebook is to convert Rico's hierarchical annotations into the YOLO object detection format by:

- Understanding the YOLO annotation format
- Identifying UI components to detect
- Extracting bounding boxes from Rico JSON files
- Mapping UI components to YOLO class IDs
- Generating YOLO label (.txt) files
- Preparing the dataset for YOLO26 training


YOLO LOOKS LIKE THIS : 
```
dataset/
│
├── images/
│   ├── train/
│   ├── val/
│
├── labels/
│   ├── train/
│   ├── val/
│
└── dataset.yaml
```

## Understanding Rico Bounding Boxes

Each UI component in the Rico dataset is represented by a bounding box.

A bounding box consists of four values:

```
[x1, y1, x2, y2]
```

where:

- **x1, y1** represent the top-left corner.
- **x2, y2** represent the bottom-right corner.

Example:

```
[0, 84, 1440, 280]
```

This means the UI component begins at coordinate `(0, 84)` and ends at coordinate `(1440, 280)`.


In [1]:
# Imports
import json
import os
import glob
from pathlib import Path

from PIL import Image
from collections import Counter
from tqdm import tqdm

## Understanding YOLO Bounding Boxes

Unlike the Rico dataset, YOLO represents each object using its center coordinates and dimensions.

YOLO annotation format:

```
class_id center_x center_y width height
```

where

- **class_id** → Integer representing the object category.
- **center_x** → Horizontal center of the object.
- **center_y** → Vertical center of the object.
- **width** → Width of the object.
- **height** → Height of the object.

All four coordinate values are normalized by the image width and height so that they lie between **0 and 1**.

### Rico Format

```
[x1, y1, x2, y2]
```

Example

```
[0, 84, 1440, 280]
```

---

### YOLO Format

```
class_id center_x center_y width height
```

Example

```
0 0.500000 0.071094 1.000000 0.076563
```

In [2]:
bounds = [0, 84, 1440, 280]

image_width = 1440
image_height = 2560

x1, y1, x2, y2 = bounds

box_width = x2 - x1
box_height = y2 - y1

center_x = x1 + box_width / 2
center_y = y1 + box_height / 2

print("Width       :", box_width)
print("Height      :", box_height)
print("Center X    :", center_x)
print("Center Y    :", center_y)

Width       : 1440
Height      : 196
Center X    : 720.0
Center Y    : 182.0


## Coordinate Normalization

The center coordinates and dimensions calculated in the previous section are measured in pixels.

YOLO requires all values to be normalized between **0 and 1**.

Normalization is performed by dividing:

- Center X by image width
- Center Y by image height
- Bounding box width by image width
- Bounding box height by image height

This allows the same annotation to work regardless of image resolution.

In [3]:
center_x = center_x / image_width
center_y = center_y / image_height

box_width = box_width / image_width
box_height = box_height / image_height

print("Normalized Center X :", center_x)
print("Normalized Center Y :", center_y)
print("Normalized Width    :", box_width)
print("Normalized Height   :", box_height)

Normalized Center X : 0.5
Normalized Center Y : 0.07109375
Normalized Width    : 1.0
Normalized Height   : 0.0765625


## Building the Conversion Function

The previous sections demonstrated the Rico to YOLO conversion manually.

To avoid repeating the same calculations for every UI component, we encapsulate the conversion logic into a reusable function.

The function accepts:

- Rico bounding box
- Image width
- Image height

and returns:

- Normalized center coordinates
- Normalized width
- Normalized height

In [4]:
def rico_to_yolo(bounds, width, height):
    x1, x2, x3, x4 = bounds
    if x2 <= x1 or y2 <= y1:
        raise ValueError(f"Invalid bounding box: {bounds}")

    box_width = x2 - x1
    box_height = y2 - y1

    center_x = x1 + box_width / 2
    center_y = y1 + box_height / 2

    center_x /= image_width
    center_y /= image_height

    box_width /= image_width
    box_height /= image_height

    return center_x, center_y, box_width, box_height


In [5]:
# simple testing of code
bounds = [0, 84, 1440, 280]

image_width = 1440
image_height = 2560

center_x, center_y, width, height = rico_to_yolo(bounds, image_width, image_height)

print("Center X :", center_x)
print("Center Y :", center_y)
print("Width    :", width)
print("Height   :", height)

Center X : 0.029166666666666667
Center Y : 0.07109375
Width    : 0.058333333333333334
Height   : 0.0765625


## Loading a Rico JSON File

Each screenshot in the Rico dataset has an associated JSON annotation file.

The annotation contains a hierarchical representation of the Android user interface, where each UI component stores information such as:

- Component class
- Bounding box
- Child components

Before processing the entire dataset, we first load and inspect a single JSON file.

In [6]:
json_path = '../Data/Combined/10.json'

with open(json_path, 'r') as f:
    data = json.load(f)

print(type(data))
# print(data)



<class 'dict'>


## Exploring the JSON Structure and Accessing the Root Component

The Rico annotation is organized as a nested dictionary.
Before accessing specific information, we inspect the available keys.

The UI hierarchy begins with a root node. 

Every other UI component is connected to this root through parent-child relationships.
By extracting the root node, we obtain the starting point for traversing the entire interface.

In [7]:
print(data.keys())
print(data.values())

root = data["activity"]["root"]

print(root["class"])

dict_keys(['activity_name', 'activity', 'is_keyboard_deployed', 'request_id'])
dict_values(['se.perigee.android.seven/com.perigee.seven.ui.activity.MainActivity', {'root': {'scrollable-horizontal': False, 'draw': True, 'ancestors': ['android.widget.FrameLayout', 'android.view.ViewGroup', 'android.view.View', 'java.lang.Object'], 'clickable': False, 'pressed': 'not_pressed', 'focusable': False, 'long-clickable': False, 'enabled': True, 'bounds': [0, 0, 1440, 2560], 'visibility': 'visible', 'content-desc': [None], 'rel-bounds': [0, 0, 1440, 2560], 'focused': False, 'selected': False, 'scrollable-vertical': False, 'children': [{'scrollable-horizontal': False, 'draw': True, 'ancestors': ['android.view.ViewGroup', 'android.view.View', 'java.lang.Object'], 'clickable': False, 'pressed': 'not_pressed', 'focusable': False, 'long-clickable': False, 'enabled': True, 'bounds': [0, 0, 1440, 2392], 'visibility': 'visible', 'content-desc': [None], 'rel-bounds': [0, 0, 1440, 2392], 'focused': False, 

## Inspecting the Root Node

Each UI component is represented as a dictionary containing multiple attributes.

These attributes describe the component's type, position, size, and child elements.

#### Among these attributes, three are especially important for this project:
- class → Identifies the UI component.
- bounds → Stores the bounding box coordinates.
- children → Contains nested UI components.
- These three fields are sufficient to generate YOLO annotations.

In [8]:
print(root.keys())


dict_keys(['scrollable-horizontal', 'draw', 'ancestors', 'clickable', 'pressed', 'focusable', 'long-clickable', 'enabled', 'bounds', 'visibility', 'content-desc', 'rel-bounds', 'focused', 'selected', 'scrollable-vertical', 'children', 'adapter-view', 'abs-pos', 'pointer', 'class', 'visible-to-user'])


In [9]:
print("Class :", root["class"])
print("Bounds:", root["bounds"])
print("Number of Children:", len(root.get("children", [])))

Class : com.android.internal.policy.PhoneWindow$DecorView
Bounds: [0, 0, 1440, 2560]
Number of Children: 3


## Understanding the UI Tree

The Rico dataset represents the Android interface as a hierarchical tree.

Each UI component is connected through parent-child relationships.

For example:
PhoneWindow$DecorView
->
LinearLayout
->
FrameLayout
->
Toolbar
->
ImageButton
->
TextView
->
RecyclerView
->
CardView
->
ImageView

Instead of storing components independently, Rico nests UI elements inside one another.

Therefore, traversing the UI hierarchy requires recursively visiting every child component.

In [10]:
children = root.get('children', [])
print(children)
print('-'*10)

for i, child in enumerate(children,start=0):
    print('child',i)
    print("Class :", child["class"])
    print("Bounds:", child["bounds"])
    print("Number of Children:", len(child.get("children", [])))
    print('-'*10)

[{'scrollable-horizontal': False, 'draw': True, 'ancestors': ['android.view.ViewGroup', 'android.view.View', 'java.lang.Object'], 'clickable': False, 'pressed': 'not_pressed', 'focusable': False, 'long-clickable': False, 'enabled': True, 'bounds': [0, 0, 1440, 2392], 'visibility': 'visible', 'content-desc': [None], 'rel-bounds': [0, 0, 1440, 2392], 'focused': False, 'selected': False, 'scrollable-vertical': False, 'children': [{'resource-id': 'android:id/action_mode_bar_stub', 'adapter-view': False, 'pointer': 'e0e434f', 'scrollable-horizontal': False, 'ancestors': ['android.view.View', 'java.lang.Object'], 'selected': False, 'content-desc': [None], 'rel-bounds': [0, 0, 0, 0], 'draw': True, 'focusable': False, 'long-clickable': False, 'visibility': 'gone', 'focused': False, 'clickable': False, 'abs-pos': True, 'class': 'android.view.ViewStub', 'visible-to-user': False, 'package': 'android', 'enabled': True, 'bounds': [0, 0, 0, 0], 'pressed': 'not_pressed', 'scrollable-vertical': False}

## Traversing the UI Tree

Since each UI component may contain additional child components, we recursively visit every node in the hierarchy.

The traversal begins at the root node.

For every node:

1. Display its class.
2. Visit each child component.

This continues until no more children remain.

In [11]:
print(root["class"])
print(root["bounds"])
print(root["children"])

com.android.internal.policy.PhoneWindow$DecorView
[0, 0, 1440, 2560]
[{'scrollable-horizontal': False, 'draw': True, 'ancestors': ['android.view.ViewGroup', 'android.view.View', 'java.lang.Object'], 'clickable': False, 'pressed': 'not_pressed', 'focusable': False, 'long-clickable': False, 'enabled': True, 'bounds': [0, 0, 1440, 2392], 'visibility': 'visible', 'content-desc': [None], 'rel-bounds': [0, 0, 1440, 2392], 'focused': False, 'selected': False, 'scrollable-vertical': False, 'children': [{'resource-id': 'android:id/action_mode_bar_stub', 'adapter-view': False, 'pointer': 'e0e434f', 'scrollable-horizontal': False, 'ancestors': ['android.view.View', 'java.lang.Object'], 'selected': False, 'content-desc': [None], 'rel-bounds': [0, 0, 0, 0], 'draw': True, 'focusable': False, 'long-clickable': False, 'visibility': 'gone', 'focused': False, 'clickable': False, 'abs-pos': True, 'class': 'android.view.ViewStub', 'visible-to-user': False, 'package': 'android', 'enabled': True, 'bounds': 

In [12]:
def traverse(node, lvl=0):
    print(' ', lvl, node['class'])
    for child in node.get("children", []):
        traverse(child, lvl + 1)

traverse(root)

  0 com.android.internal.policy.PhoneWindow$DecorView
  1 android.widget.LinearLayout
  2 android.view.ViewStub
  2 android.widget.FrameLayout
  3 android.support.v7.widget.ActionBarOverlayLayout
  4 android.support.v7.widget.ContentFrameLayout
  5 android.support.design.widget.CoordinatorLayout
  6 android.widget.FrameLayout
  7 android.support.design.widget.CoordinatorLayout
  8 android.support.v7.widget.RecyclerView
  9 android.widget.LinearLayout
  10 android.support.v7.widget.AppCompatTextView
  9 android.widget.LinearLayout
  10 android.support.v7.widget.AppCompatImageView
  10 android.support.v7.widget.AppCompatTextView
  9 android.widget.LinearLayout
  10 com.perigee.seven.ui.view.SevenSquareCardView
  11 android.widget.RelativeLayout
  12 android.widget.FrameLayout
  13 android.support.v7.widget.AppCompatImageView
  12 android.widget.FrameLayout
  13 android.support.v7.widget.AppCompatTextView
  12 android.support.v7.widget.AppCompatImageView
  9 android.widget.LinearLayout
  

## Selecting Relevant UI Components

The Rico dataset contains hundreds of Android component types.

Many of these components are layout containers that organize the interface but are not visually meaningful for object detection.

Therefore, only a subset of UI components is selected for YOLO training.

The selected components are:

- Toolbar
- Text
- Image
- Image Button
- Button
- Card View
- Recycler View

All remaining components are ignored during preprocessing.
## Filtering Components

Each UI component encountered during traversal is checked against the predefined class mapping.

Only components present in the class mapping are retained.

All other components are ignored.

In [13]:
CLASS_MAP = {
    "Toolbar": 0,
    "AppCompatTextView": 1,
    "RobotoTextView": 1,
    "AppCompatImageView": 2,
    "ImageView": 2,
    "ImageButton": 3,
    "AppCompatButton": 4,
    "CardView": 5,
    "RecyclerView": 6,
}

def should_keep(node):
    if node is None:
        return False
    if "class" not in node:
        return False
    if node["class"] is None:
        return False
    class_name=node["class"].split(".")[-1]
    return class_name in CLASS_MAP


## Generating YOLO Labels

The UI tree is traversed recursively.

For every UI component:

1. Check whether the component should be retained.
2. Convert its Rico bounding box into YOLO format.
3. Assign the corresponding class ID.
4. Store the annotation.

The collected annotations will later be written into a YOLO label file.

In [14]:
yolo_labels=[]

def traverse_keep(node):
    global yolo_labels
    if node is None:
        return
    if should_keep(node):
        class_name=node["class"].split(".")[-1]
        class_id=CLASS_MAP[class_name]
        try:
            center_x, center_y, width, height = rico_to_yolo(node["bounds"], image_width, image_height)
            yolo_labels.append(f"{class_id} {center_x:.6f} {center_y:.6f} {width:.6f} {height:.6f}")
        except (ValueError, KeyError, TypeError):
            pass
    for child in node.get("children", []):
        traverse_keep(child)


In [15]:
traverse_keep(root)
print("Labels Generated:", len(yolo_labels))


Labels Generated: 40


In [16]:
with open("10.txt", "w") as f:
    for label in yolo_labels:
        f.write(label + "\n")

In [17]:
# folder_path = Path("../Data/Combined/")

# # Get all JSON files and slice the list to the first 10
# json_files = sorted(folder_path.glob("*.json"))[0:10]

# for file_path in json_files:
#     with open(file_path, "r") as f:
#         data = json.load(f)
#         print(f"--- File: {file_path.name} ---")

In [18]:
output_folder = "../labels"

os.makedirs(output_folder, exist_ok=True)

In [19]:
file_name = os.path.splitext(os.path.basename(json_path))[0]


output_file = os.path.join(output_folder, file_name + ".txt")
print(output_file)

../labels/10.txt


In [20]:
with open(output_file, "w") as f:
    for label in yolo_labels:
        f.write(label + "\n")

## Processing a Single Rico Annotation

The preprocessing steps developed in the previous sections are combined into a reusable function.

The function performs the complete workflow:

1. Load a Rico JSON annotation.
2. Extract the UI hierarchy.
3. Generate YOLO annotations.
4. Save the corresponding label file.

This function can then be applied to every JSON file in the dataset.

In [21]:
def process_json(json_path, output_dir):
    global yolo_labels

    with open(json_path, "r") as f:
        data = json.load(f)

    root = data["activity"]["root"]

    yolo_labels = []

    traverse_keep(root)

    file_name = os.path.splitext(os.path.basename(json_path))[0]

    output_file = os.path.join(output_dir, file_name + ".txt")

    with open(output_file, "w") as f:
        for label in yolo_labels:
            f.write(label + "\n")

    return len(yolo_labels)

In [22]:
labels = process_json(json_path, output_folder)

print(f"Generated {labels} labels")

Generated 40 labels


## Processing the Entire Rico Dataset

The `process_json()` function is applied to every Rico annotation file.

For each JSON file:

1. Load the annotation.
2. Traverse the UI hierarchy.
3. Generate YOLO labels.
4. Save the corresponding `.txt` annotation.

A progress bar is displayed to monitor preprocessing.

In [26]:
import glob
import os

json_folder = "..Data/Combined"  # <-- change this if your JSON folder is different

json_files = sorted(glob.glob("../Data/Combined/*.json"))

print(f"Found {len(json_files)} JSON files")
for json_file in json_files:
    process_json(json_file, output_folder)

Found 66261 JSON files


In [31]:
label_files = glob.glob("../labels/*.txt")

print(f"Total label files: {len(label_files)}")

Total label files: 66261
